# 第 19 节：PPO 理论 (Proximal Policy Optimization)

---

## 📍 本节位置

```
Actor-Critic (14) → 重要性采样 (16) → GAE (18) → **PPO 理论 (19)** → PPO 实战 (20)
                                                       ↑
                                                   你在这里
```

PPO 是 OpenAI 在 2017 年提出的策略梯度算法，它通过裁剪目标函数来限制策略更新幅度，在 TRPO 的基础上大幅简化了实现，成为当今最流行的深度强化学习算法之一。

> Schulman, J., Wolski, F., Dhariwal, P., Radford, A., & Klimov, O. (2017). Proximal Policy Optimization Algorithms. *arXiv:1707.06347*.


## 🎯 学习目标

1. 理解 TRPO 的局限性以及 PPO 的改进动机
2. 掌握 PPO 的 Clipped Surrogate Objective 公式及其几何意义
3. 分析裁剪行为的四种情况（好/坏动作 × 超/欠更新）
4. 对比 PPO-Clip 和 PPO-Penalty 两种变体
5. 理解为什么 PPO 可以安全地使用多轮小批量更新
6. 学会解读 PPO 的监控指标（clip fraction, approx KL, explained variance）


## 从 TRPO 到 PPO：动机

### TRPO 的贡献

Trust Region Policy Optimization (TRPO, Schulman et al., 2015) 首次提出使用**信任区域**来保证策略更新的单调改进：

$$\max_\theta \; \mathbb{E}\left[\frac{\pi_\theta(a \mid s)}{\pi_{\theta_{\text{old}}}(a \mid s)} A^{\pi_{\theta_{\text{old}}}}(s, a)\right]$$

$$\text{s.t.} \; \mathbb{E}\left[\text{KL}\left[\pi_{\theta_{\text{old}}}(\cdot \mid s) \;||\; \pi_\theta(\cdot \mid s)\right]\right] \leq \delta$$

### TRPO 的问题

| 问题 | 描述 | 影响 |
|:----:|:----|:-----|
| **计算复杂** | 需要计算 Fisher 信息矩阵 + 共轭梯度 + 线搜索 | 每次更新的计算量大 |
| **实现困难** | 自然梯度的近似需要大量数值技巧 | 代码易出错，调试困难 |
| **二阶信息** | 使用 KL 散度的二阶近似（Fisher 矩阵） | 近似误差难以控制 |
| **兼容性** | 难以与 CNN 或 RNN + 共享参数架构配合 | 限制了应用范围 |

### PPO 的核心洞察

> "既然约束策略更新这么复杂，我们为什么不直接把更新幅度限制在目标函数里？"

PPO 放弃了显式的 KL 约束，转而使用一个**裁剪的目标函数**，当策略更新超过预设范围时自动惩罚过大的更新。这使得 PPO：

- 只需要一阶梯度的信息（不需要 Fisher 矩阵）
- 实现简单（几十行 PyTorch 代码）
- 计算效率高（与标准策略梯度一样快）
- 与 TRPO 一样稳定，甚至更优


## PPO 的核心思想：用裁剪替代约束

### 标准策略梯度目标

标准策略梯度最大化：

$$L^{\text{PG}}(\theta) = \mathbb{E}_t\left[\log \pi_\theta(a_t \mid s_t) \, A_t\right]$$

问题：单步更新可能大幅改变策略，导致性能崩塌。

### PPO 的思路

引入**重要性采样比率** $r_t(\theta)$ 来衡量策略变化幅度：

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

当 $\pi_\theta = \pi_{\theta_{\text{old}}}$ 时，$r_t(\theta) = 1$。

PPO 的目标：**最大化收益的同时，防止 $r_t(\theta)$ 偏离 1 太远**。

### 两种实现方式

| 变体 | 方法 | 优点 |
|:----:|:----|:-----|
| **PPO-Clip** | 在目标函数中直接裁剪 $r_t(\theta)$ | 简单，无需辅助计算 |
| **PPO-Penalty** | 在目标函数中加入 KL 惩罚项 | 有 TRPO 的理轮基础 |

> 实践中，PPO-Clip 更常用，因为它不需要调整 KL 惩罚系数。

### 为什么叫做 "Proximal" (近端)？

> PPO 确保每次更新后的新策略 $\pi_\theta$ 在**近端**（即与旧策略 $\pi_{\theta_{\text{old}}}$ 相近的位置），而不是一步跳到远端。


## Surrogate Objective（替代目标函数）

### 重要性采样形式的策略梯度

从重要性采样出发，我们想要最大化新策略下的期望回报，但使用旧策略收集的数据：

$$J(\theta) = \mathbb{E}_{a \sim \pi_{\theta_{\text{old}}}}\left[\frac{\pi_\theta(a \mid s)}{\pi_{\theta_{\text{old}}}(a \mid s)} \, A^{\pi_{\theta_{\text{old}}}}(s, a)\right]$$

### 定义比率

$$r_t(\theta) = \frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)}$$

当 $\theta = \theta_{\text{old}}$ 时，$r_t(\theta) = 1$。

### Surrogate Objective

$$L^{\text{CPI}}(\theta) = \mathbb{E}_t\left[r_t(\theta) \, A_t\right]$$

其中 CPI 代表 "Conservative Policy Iteration"。

### 如果没有约束会怎样？

最大化 $L^{\text{CPI}}$ 时，如果某个动作的优势 $A_t > 0$，策略会无限增大 $\pi_\theta(a_t \mid s_t)$（即 $r_t(\theta) \to \infty$）来提高目标函数。

**这会导致灾难**：
- 策略坍缩到单一动作
- 失去探索能力
- 如果该动作实际上是次优的（优势估计有误差），性能会大幅下降

### PPO 的解决方案

> PPO 通过限制 $r_t(\theta)$ 的变化范围来解决这个问题——要么通过裁剪（Clip），要么通过惩罚（KL Penalty）。


## PPO-Clip 目标函数

### 完整公式

$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) A_t, \; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) \, A_t\right)\right]$$

其中 $\varepsilon$ 是裁剪范围（通常 $\varepsilon = 0.1$ 或 $0.2$）。

### 词汇解释

| 符号 | 含义 | 典型值 |
|:----:|:----|:------:|
| $r_t(\theta)$ | 重要性采样比率 | $\approx 1$（初值）|
| $A_t$ | GAE 优势估计 | 归一化后 $\sim \mathcal{N}(0,1)$ |
| $\varepsilon$ | 裁剪范围 | $0.1 \sim 0.2$ |
| $\text{clip}(r, 1-\varepsilon, 1+\varepsilon)$ | 将 $r$ 裁剪到 $[1-\varepsilon, 1+\varepsilon]$ 区间内 | $[0.8, 1.2]$（$\varepsilon=0.2$）|

### 总损失函数

完整的 PPO 损失包含三个部分：

$$L_t(\theta) = \mathbb{E}_t\left[L^{\text{CLIP}}(\theta) - c_1 L^{\text{VF}}(\theta) + c_2 H(\pi_\theta(\cdot \mid s_t))\right]$$

- $L^{\text{CLIP}}(\theta)$: 裁剪的策略损失（最大化期望回报）
- $L^{\text{VF}}(\theta)$: 价值函数损失（通常使用裁剪后的 MSE）
- $H(\pi_\theta(\cdot \mid s_t))$: 策略熵（鼓励探索）
- $c_1, c_2$: 各项的权重系数（典型值 $c_1 = 0.5, c_2 = 0.01$）


## 裁剪行为的深入分析

### 四种情况

裁剪的行为取决于 $A_t$ 的正负和 $r_t(\theta)$ 与 1 的关系：

```
                A > 0 (好动作)              A < 0 (坏动作)
                ─────────────              ─────────────

r > 1+ε        ① 动作更好，但              ② 动作更差，但
                更新太猛 → 裁剪              策略概率增加 → 保留
                损失 = (1+ε)·A              损失 = r·A (负值更大)

r < 1-ε        ③ 动作更好，但              ④ 动作更差，但
                策略概率降低 → 保留           更新太猛 → 裁剪
                损失 = r·A (正值更小)        损失 = (1-ε)·A
```

### 详细分析

#### 情况 ①：$A > 0$ 且 $r > 1+\varepsilon$（好动作，过度更新）

- **语义**：这个动作很好，而且新策略已经大幅提高了它的概率
- **裁剪效果**：将 $r$ 裁剪为 $1+\varepsilon$，防止进一步提高
- **原因**：已经加了很多了，再加大可能会过犹不及
- **直觉**："这个动作不错，但你改太多了，停下来"

#### 情况 ②：$A < 0$ 且 $r > 1+\varepsilon$（坏动作，错误增加）

- **语义**：这个动作不好，但新策略反而增加了它的概率
- **裁剪效果**：保留 $r \cdot A$（这是一个很大的负数，强烈的惩罚信号）
- **原因**：需要惩罚这种错误的方向变化
- **直觉**："这个动作更差的，你怎么还增加了它的概率？大大的惩罚！"

#### 情况 ③：$A > 0$ 且 $r < 1-\varepsilon$（好动作，反向减少）

- **语义**：这个动作很好，但新策略降低了它的概率
- **裁剪效果**：保留 $r \cdot A$（这是一个小的正数，微弱奖励）
- **原因**：减少好动作的概率不是我们想要的，但没必要猛烈惩罚
- **直觉**："这个动作不错，但你降低了它的概率？好吧，那就少奖励一点"

#### 情况 ④：$A < 0$ 且 $r < 1-\varepsilon$（坏动作，过度减少）

- **语义**：这个动作不好，新策略大幅降低了它的概率
- **裁剪效果**：将 $r$ 裁剪为 $1-\varepsilon$，防止进一步降低
- **原因**：已经降很多了，再降可能会影响探索
- **直觉**："这个动作不好，但你已经降很多了，就这样吧"

### 关键洞察

> 裁剪是**非对称**的！它对好动作增加概率和坏动作减少概率都施加了限制，但对相反方向（好动作减少概率、坏动作增加概率）保留了大惩罚。这意味着裁剪就像信任区域的一个**单向屏障**——它允许策略变差时的大幅惩罚，但阻止策略过度优化。


In [ ]:
# ============================================================
# 可视化 PPO 裁剪函数
# ============================================================
# 绘制 L^CLIP vs ratio，分别对 A>0 和 A<0

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epsilon = 0.2
ratios = np.linspace(0.0, 2.5, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A > 0 的情况
A_pos = 1.0
surr1_pos = ratios * A_pos
surr2_pos = np.clip(ratios, 1 - epsilon, 1 + epsilon) * A_pos
L_clip_pos = np.minimum(surr1_pos, surr2_pos)

ax = axes[0]
ax.plot(ratios, surr1_pos, 'b--', alpha=0.5, label='r·A (unclipped)')
ax.plot(ratios, surr2_pos, 'g--', alpha=0.5, label='clip(r)·A')
ax.plot(ratios, L_clip_pos, 'r-', linewidth=2.5, label='L^CLIP = min(...)')
ax.axvline(1 - epsilon, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1 + epsilon, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1.0, color='black', linestyle='-', alpha=0.3)
ax.fill_between(ratios, 0, L_clip_pos, where=(ratios >= 1-epsilon) & (ratios <= 1+epsilon),
                color='green', alpha=0.1, label='信任区域')
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('L^CLIP')
ax.set_title(f'A > 0 (好动作), ε = {epsilon}')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2.5)

# A < 0 的情况
A_neg = -1.0
surr1_neg = ratios * A_neg
surr2_neg = np.clip(ratios, 1 - epsilon, 1 + epsilon) * A_neg
L_clip_neg = np.minimum(surr1_neg, surr2_neg)

ax = axes[1]
ax.plot(ratios, surr1_neg, 'b--', alpha=0.5, label='r·A (unclipped)')
ax.plot(ratios, surr2_neg, 'g--', alpha=0.5, label='clip(r)·A')
ax.plot(ratios, L_clip_neg, 'r-', linewidth=2.5, label='L^CLIP = min(...)')
ax.axvline(1 - epsilon, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1 + epsilon, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1.0, color='black', linestyle='-', alpha=0.3)
ax.fill_between(ratios, 0, L_clip_neg, where=(ratios >= 1-epsilon) & (ratios <= 1+epsilon),
                color='green', alpha=0.1, label='信任区域')
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('L^CLIP')
ax.set_title(f'A < 0 (坏动作), ε = {epsilon}')
ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
ax.set_xlim(0, 2.5)

plt.suptitle('PPO-Clip 目标函数：r 和 A 的四种组合', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_clip_function.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_clip_function.png")

# 额外：显示裁剪的 4 个区域
print("=== 裁剪分析表 ===")
print(f"{'区域':<12} {'条件':<28} {'最终损失':<20}")
print("-" * 60)
print(f"{'① 好·超调':<12} {'A>0 AND r>1+ε':<28} {'r_clipped·A = (1+ε)A':<20}")
print(f"{'② 坏·错增':<12} {'A<0 AND r>1+ε':<28} {'r·A (保留大的负值)':<20}")
print(f"{'③ 好·误减':<12} {'A>0 AND r<1-ε':<28} {'r·A (保留小的正值)':<20}")
print(f"{'④ 坏·过度':<12} {'A<0 AND r<1-ε':<28} {'r_clipped·A = (1-ε)A':<20}")


In [ ]:
# ============================================================
# 不同 ε 对 PPO 裁剪函数的影响
# ============================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

epsilons = [0.0, 0.1, 0.2, 0.5]
ratios = np.linspace(0.0, 3.0, 500)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# A > 0
A_pos = 1.0
ax = axes[0]
for eps in epsilons:
    surr1 = ratios * A_pos
    surr2 = np.clip(ratios, 1 - eps, 1 + eps) * A_pos
    L_clip = np.minimum(surr1, surr2)
    ax.plot(ratios, L_clip, linewidth=1.5, label=f'ε={eps}')
ax.axvline(1.0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('L^CLIP')
ax.set_title('A > 0：不同 ε 下的裁剪效果')
ax.legend(); ax.grid(True, alpha=0.3)

# A < 0
A_neg = -1.0
ax = axes[1]
for eps in epsilons:
    surr1 = ratios * A_neg
    surr2 = np.clip(ratios, 1 - eps, 1 + eps) * A_neg
    L_clip = np.minimum(surr1, surr2)
    ax.plot(ratios, L_clip, linewidth=1.5, label=f'ε={eps}')
ax.axvline(1.0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('L^CLIP')
ax.set_title('A < 0：不同 ε 下的裁剪效果')
ax.legend(); ax.grid(True, alpha=0.3)

plt.suptitle('PPO 裁剪对 ε 的敏感性分析', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_epsilon_sensitivity.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_epsilon_sensitivity.png")

print("=== ε 极端值分析 ===")
print(f"ε=0.0: 退化为 min(rA, A) — 当 A>0 时 L=r·A 但 r<1 被限制; 当 A<0 时 L=r·A 但 r>1 被限制")
print(f"        等价于 L = min(rA, A)，这是不可行的")
print(f"ε=0.5: 软约束，允许较大更新，但仍有上限")
print(f"ε→∞:  退化为标准策略梯度 L = r·A（无约束）")


## PPO-Penalty：基于 KL 惩罚的变体

### 目标函数

除了裁剪，PPO 还提出了另一种实现方式——在目标函数中添加 KL 散度惩罚项：

$$L^{\text{KLPEN}}(\theta) = \mathbb{E}_t\left[\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)} A_t\right] - \beta \cdot \text{KL}\left[\pi_{\theta_{\text{old}}}(\cdot \mid s_t) \;||\; \pi_\theta(\cdot \mid s_t)\right]$$

### 自适应调整 $\beta$

PPO-Penalty 的核心挑战是选择合适的 $\beta$。PPO 论文提出了自适应的调整策略：

```
目标 KL = d_targ (例如 0.01)

每次更新后:
    if KL > d_targ * 1.5:  β ← β * 2     # KL 太大，加大惩罚
    if KL < d_targ / 1.5:  β ← β / 2     # KL 太小，减小惩罚
```

### PPO-Clip vs PPO-Penalty

| 特性 | PPO-Clip | PPO-Penalty |
|:----:|:--------:|:-----------:|
| 约束方式 | 隐式（裁剪）| 显式（KL 惩罚）|
| 超参数 | $\varepsilon$（裁剪范围）| $\beta$（惩罚系数）|
| 自适应 | 不需要 | 需要（KL 目标值）|
| 实现复杂度 | 低 | 中 |
| 稳定性 | 对 $\varepsilon$ 不敏感 | 对 $\beta$ 敏感 |
| 理论基础 | 启发式 | 有 TRPO 理论支撑 |

### 实践经验

> 大多数开源实现（Stable-Baselines3, rllib, CleanRL 等）默认使用 PPO-Clip。PPO-Penalty 在需要更精细控制的场景（如机器人控制）中偶有使用。


## 为什么裁剪不是信任区域？

### 信任区域的本质

一个真正的信任区域方法需要确保**在整个状态空间上**，新旧策略的差异（以 KL 散度衡量）被限制在某个范围内。

TRPO 的约束：
$$\mathbb{E}_s\left[\text{KL}[\pi_{\theta_{\text{old}}}(\cdot \mid s) \;||\; \pi_\theta(\cdot \mid s)]\right] \leq \delta$$

这是**全局约束**——它同时约束所有状态的策略变化。

### 裁剪的局限性

PPO 的裁剪 $\text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon)$ 是**逐动作局部约束**：

1. **单向限制**：裁剪只限制 $r_t(\theta)$ 在 $[1-\varepsilon, 1+\varepsilon]$ 范围内，但不限制 $\pi_\theta$ 本身的变化
2. **无全局视图**：不考虑状态空间中不同区域的累积变化
3. **无 KL 约束**：即使 $r_t(\theta)$ 被裁剪，KL 散度仍可能很大

### 为什么 PPO 仍然有效？

PPO 虽然不能完美保证信任区域，但在实践中通过以下机制实现了类似的稳定性：

| 机制 | 作用 |
|:----:|:-----|
| **裁剪** | 直接阻止单步更新过大 |
| **多 epoch 更新** | 即使被裁剪，多次更新也能逐步改进 |
| **KL 提前停止** | 可选的安全机制，监控 KL 散度 |
| **Advantage 归一化** | 稳定梯度尺度 |
| **小的学习率** | 配合裁剪，进一步限制步长 |

### 形象比喻

> TRPO 像是一条带**防护栏**的道路——你必须保持在车道内。
> PPO 像是在路面画了**减速带**——你可以在车道上自由行驶，但每经过一个减速带都会颠簸一下，自然地限制了你偏离车道的程度。

TRPO 提供的是**硬约束**（必须满足），PPO 提供的是**软约束**（鼓励满足）。


## 多 Epoch 更新：数据复用的艺术

### 标准策略梯度的限制

在标准的 REINFORCE 中，每条轨迹只能使用一次：
$$\theta \leftarrow \theta + \alpha \nabla_\theta \log \pi_\theta(a_t \mid s_t) G_t$$

数据效率低——收集 2048 步，更新一次，丢弃数据。

### PPO 的多 epoch 更新

PPO 在每次 rollout 后对同一批数据执行多次 minibatch SGD 更新：

```
收集 n_steps 步数据
┌─────────────────────────────┐
│  Epoch 1: SGD with minibatches  │
│  Epoch 2: SGD with minibatches  │
│  ...                           │
│  Epoch N: SGD with minibatches  │
└─────────────────────────────┘
丢弃数据，收集新的 rollout
```

### 为什么可以复用数据？

关键在于**重要性比率 $r_t(\theta)$**：

$$L(\theta) = \mathbb{E}_{\theta_{\text{old}}}\left[r_t(\theta) \, A_t\right]$$

- 在第一次更新时，$\theta = \theta_{\text{old}}$，$r=1$
- 随着更新进行，$\theta$ 偏离 $\theta_{\text{old}}$，$r$ 偏离 1
- **裁剪** 防止 $r$ 偏离太远，从而保证目标函数仍然是一个合理的近似

### 直觉

> 如果每次更新改变一点点（裁剪限制），那么即使更新 10 次，总的 KL 散度也可能只有 0.1 左右。这等价于：我们在一个小的信任区域内完成了多次梯度步骤，而不是一次大步更新。

### 典型配置

| 参数 | 典型值 | 说明 |
|:----:|:------:|:------|
| `n_steps` | 2048 | 每次 rollout 收集的步数 |
| `batch_size` | 64 | minibatch 大小 |
| `n_epochs` | 10 | 每个 rollout 的训练轮数 |
| `clip_epsilon` | 0.2 | 裁剪范围 |


In [ ]:
# ============================================================
# PPO Surrogate Loss 地貌可视化
# ============================================================
# 展示 PPO 的 L^CLIP 随比率 r 和优势 A 的变化

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# 创建比率和优势的网格
eps = 0.2
ratios = np.linspace(0.0, 2.5, 100)
advantages = np.linspace(-2.0, 2.0, 100)
R, A = np.meshgrid(ratios, advantages)

# 计算 L^CLIP
surr1 = R * A
surr2 = np.clip(R, 1 - eps, 1 + eps) * A
L_clip = np.minimum(surr1, surr2)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 2D 热力图
im = axes[0].contourf(R, A, L_clip, levels=20, cmap='RdBu_r')
axes[0].contour(R, A, L_clip, levels=10, colors='black', linewidths=0.3)
axes[0].axvline(1 - eps, color='green', linestyle='--', label=f'1-ε={1-eps}')
axes[0].axvline(1 + eps, color='green', linestyle='--', label=f'1+ε={1+eps}')
axes[0].axhline(0, color='gray', linestyle='-', alpha=0.5)
axes[0].axvline(1.0, color='gray', linestyle='-', alpha=0.5)
axes[0].set_xlabel('比率 r_t(θ)'); axes[0].set_ylabel('优势 A_t')
axes[0].set_title(f'L^CLIP 地貌 (ε={eps})')
plt.colorbar(im, ax=axes[0], label='L^CLIP')

# 3D 视角
ax = axes[1]
im2 = ax.imshow(L_clip, extent=[0, 2.5, -2, 2], origin='lower',
                aspect='auto', cmap='RdBu_r')
ax.axvline(1 - eps, color='green', linestyle='--', alpha=0.7)
ax.axvline(1 + eps, color='green', linestyle='--', alpha=0.7)
ax.axhline(0, color='gray', linestyle='-', alpha=0.5)
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('优势 A_t')
ax.set_title('L^CLIP 颜色映射')
plt.colorbar(im2, ax=ax, label='L^CLIP')

plt.suptitle('PPO 替代目标函数地貌: L^CLIP(r, A)', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_loss_landscape.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_loss_landscape.png")

print("说明:")
print("  - 绿色虚线: 裁剪边界 [1-ε, 1+ε]")
print("  - 在边界内: L = r·A (标准 IS)")
print("  - 在边界外: L 被裁剪，梯度被限制")


In [ ]:
# ============================================================
# 演示 PPO 训练中比率分布的变化
# ============================================================
# 模拟 PPO 的多 epoch 更新，观察 r_t(θ) 分布的变化
# 以及裁剪的效果

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 模拟一个策略更新过程
n_samples = 10000
n_epochs = 10

# 初始比率 ~ N(1, 0.02) — 更新前集中在 1 附近
# 随着更新进行，比率向两边扩散
# 裁剪会阻止比率超过 [1-ε, 1+ε]

epsilon = 0.2
clip_low = 1.0 - epsilon
clip_high = 1.0 + epsilon

# 模拟每一轮的比率分布（逐渐发散）
fig, axes = plt.subplots(2, 1, figsize=(12, 10))

# 扩散因子（模拟策略更新导致的比率发散）
spread_factors = np.linspace(0.02, 0.30, n_epochs)

clip_fractions = []
for epoch in range(n_epochs):
    # 比率分布以 1 为中心，标准差逐渐增大
    ratios = np.random.lognormal(
        mean=-0.5 * spread_factors[epoch]**2,
        sigma=spread_factors[epoch],
        size=n_samples
    )
    # 裁剪
    ratios_clipped = np.clip(ratios, clip_low, clip_high)
    clip_frac = np.mean((ratios < clip_low) | (ratios > clip_high))
    clip_fractions.append(clip_frac)

    axes[0].hist(ratios, bins=80, alpha=0.3, density=True,
                 label=f'Epoch {epoch+1}' if epoch in [0, 3, 6, 9] else '')

# 标记裁剪边界
axes[0].axvline(clip_low, color='red', linestyle='--', linewidth=1.5,
                label=f'裁剪下限 1-ε={clip_low}')
axes[0].axvline(clip_high, color='red', linestyle='--', linewidth=1.5,
                label=f'裁剪上限 1+ε={clip_high}')
axes[0].axvline(1.0, color='black', linestyle='-', alpha=0.3)
axes[0].set_xlabel('比率 r_t(θ)'); axes[0].set_ylabel('密度')
axes[0].set_title(f'PPO 多 Epoch 更新中的比率分布变化 (ε={epsilon})')
axes[0].legend(fontsize=8); axes[0].grid(True, alpha=0.3)
axes[0].set_xlim(0.2, 1.8)

# 裁剪分数变化
axes[1].plot(range(1, n_epochs + 1), clip_fractions, 'bo-', linewidth=2, markersize=8)
axes[1].axhline(0.1, color='gray', linestyle='--', alpha=0.5, label='10% 裁剪警告线')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('裁剪分数 (Clip Fraction)')
axes[1].set_title('裁剪分数随训练 Epoch 的变化')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_ratio_distribution.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_ratio_distribution.png")

print("=== 各 epoch 的裁剪分数 ===")
for epoch, cf in enumerate(clip_fractions):
    marker = " ⚠️ 过高！" if cf > 0.25 else ""
    print(f"Epoch {epoch+1:2d}: 裁剪分数 = {cf:.4f}{marker}")
print(f"\n说明：裁剪分数 > 0.25 意味着超过 25% 的样本被裁剪，"
      f"可能意味着策略更新过快或 ε 太小")


## 常见 PPO 监控指标

### 1. Clip Fraction（裁剪分数）

$$\text{clip\_fraction} = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}[r_i < 1-\varepsilon \;\text{or}\; r_i > 1+\varepsilon]$$

- **含义**：被裁剪的样本比例
- **经验参考** (高度依赖环境和实现)：0.05 - 0.25
- **过低**（< 0.01）：裁剪从未激活，可能 $\varepsilon$ 太大或更新幅度太小
- **过高**（> 0.5）：大部分样本被裁剪，可能 $\varepsilon$ 太小或学习率太大

### 2. Approx KL（近似 KL 散度）

$$\text{KL}_{\text{approx}} = \frac{1}{2} \cdot \mathbb{E}\left[(\log \pi_\theta - \log \pi_{\theta_{\text{old}}})^2\right]$$

- **含义**：新旧策略之间的差异
- **经验参考** (高度依赖环境和实现)：0.001 - 0.05
- **过低**（< 0.0001）：策略几乎没有更新
- **过高**（> 0.1）：策略变化过大，可能导致性能崩塌

### 3. Explained Variance（解释方差）

$$\text{EV} = 1 - \frac{\text{Var}[G_t - V(s_t)]}{\text{Var}[G_t]}$$

- **含义**：价值函数对回报的拟合质量
- **正常范围**：0.8 - 1.0（好的拟合），0.0 - 0.3（差拟合），< 0（比常数预测还差）
- **过低**：价值网络欠拟合，需要增加网络容量或减少学习率

### 诊断示例

```
Update  1:  policy_loss=-0.012  value_loss=0.483  entropy=0.693
            approx_kl=0.003    clip_frac=0.045   explained_var=0.981

Update 10:  policy_loss=-0.008  value_loss=0.127  entropy=0.685
            approx_kl=0.005    clip_frac=0.087   explained_var=0.901

Update 50:  policy_loss=-0.005  value_loss=0.051  entropy=0.568
            approx_kl=0.008    clip_frac=0.092   explained_var=0.892
```

### 预警信号

| 信号 | 可能原因 | 解决方案 |
|:----:|:---------|:---------|
| clip_frac > 0.5 | 学习率太高或 $\varepsilon$ 太小 | 降低 lr 或增大 $\varepsilon$ |
| approx_kl > 0.1 | 策略更新过快 | 降低 lr, 增加 $\varepsilon$, 启用 target_kl |
| explained_var < 0 | 价值函数完全失效 | 降低 lr, 增大网络, 检查奖励缩放 |
| entropy 快速下降 | 策略过早确定性 | 增大 $c_2$, 检查奖励设计 |


## PPO 的价值函数裁剪

### 动机

与策略网络类似，价值网络也可能出现更新幅度过大的问题。PPO 对价值损失也应用了裁剪技术：

$$L^{\text{VF}}(\theta) = \mathbb{E}_t\left[\max\left( (V_\theta(s_t) - G_t)^2, (V_{\text{clipped}}(s_t) - G_t)^2 \right)\right]$$

其中裁剪后的价值为：
$$V_{\text{clipped}}(s_t) = V_{\text{old}}(s_t) + \text{clip}\left(V_\theta(s_t) - V_{\text{old}}(s_t), -\varepsilon, +\varepsilon\right)$$

### 为什么需要价值函数裁剪？

1. **价值函数对回报的拟合需要稳定**：价值函数是优势估计的基础
2. **防止价值网络单步更新过大**：避免价值估计震荡
3. **价值更新的窗口对齐**：与策略更新保持相同的"信任区域"

### 裁剪的价值损失行为分析

| 情况 | 条件 | 效果 |
|:----:|:----:|:------|
| 未裁剪 | $|V_\theta - V_{\text{old}}| \leq \varepsilon$ | 正常 MSE 更新 |
| 裁剪激活 | $V_\theta - V_{\text{old}} > \varepsilon$ | 使用 $V_{\text{clipped}}$ 限制增长 |
| 裁剪激活 | $V_{\text{old}} - V_\theta > \varepsilon$ | 使用 $V_{\text{clipped}}$ 限制下降 |

### 实际配置

在标准实现中，价值函数裁剪的 $\varepsilon$ 通常与策略裁剪的 $\varepsilon$ 相同（默认 0.2）。有些实现（如 Stable-Baselines3）也支持独立设置价值函数的裁剪范围。

### 注意

关于价值函数裁剪的有效性在社区中存在争议。一些研究表明：
- 价值函数裁剪对最终性能的影响较小
- 价值函数损失的主要作用是为策略梯度提供稳定的优势基线
- 在某些任务上，不裁剪价值函数反而更好


In [ ]:
# ============================================================
# PPO 裁剪在动作中的可视化展示
# ============================================================
# 演示在不同 ε 值和不同优势符号下裁剪的实际效果

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

np.random.seed(42)

# 1. 模拟一个离散动作空间的策略更新
n_actions = 5

# 旧策略 logits（收集数据时的策略）
old_logits = np.array([0.5, 1.2, -0.3, 0.8, -1.0])

# 新策略 logits（更新后的策略）
new_logits = np.array([1.8, 0.6, -0.8, 0.9, -1.5])

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum()

old_probs = softmax(old_logits)
new_probs = softmax(new_logits)

print("=== 策略概率变化 ===")
print(f"{'动作':<6} {'旧 π':<12} {'新 π':<12} {'比率 r':<12}")
for a in range(n_actions):
    r = new_probs[a] / old_probs[a]
    print(f"{a:<6} {old_probs[a]:<12.4f} {new_probs[a]:<12.4f} {r:<12.4f}")

# 2. 可视化裁剪效果
epsilons = [0.1, 0.2, 0.3]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, eps in enumerate(epsilons):
    ax = axes[idx]
    actions_idx = np.arange(n_actions)
    clip_low = 1.0 - eps
    clip_high = 1.0 + eps

    # 计算原始比率
    ratios = new_probs / old_probs
    clipped_ratios = np.clip(ratios, clip_low, clip_high)

    x = np.arange(n_actions)
    width = 0.35
    ax.bar(x - width/2, ratios, width, alpha=0.7, label='原始比率 r')
    ax.bar(x + width/2, clipped_ratios, width, alpha=0.7, label=f'裁剪后 (ε={eps})')
    ax.axhline(clip_low, color='red', linestyle='--', alpha=0.5)
    ax.axhline(clip_high, color='red', linestyle='--', alpha=0.5)
    ax.axhline(1.0, color='black', linestyle='-', alpha=0.2)
    ax.set_xticks(x); ax.set_xticklabels([f'a={a}' for a in range(n_actions)])
    ax.set_ylabel('比率 r_t(θ)')
    ax.set_title(f'裁剪效果 (ε={eps})')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('PPO 裁剪对不同动作概率比率的影响', fontsize=14)
plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_clipping_action.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_clipping_action.png")

# 3. 综合可视化：展示四种情况
fig, ax = plt.subplots(figsize=(10, 6))

eps = 0.2
ratios_grid = np.linspace(0.2, 2.0, 300)

# 四种组合
A_vals = [1.0, 1.0, -1.0, -1.0]
r_ranges = [(1.2, 2.0), (0.2, 0.8), (1.2, 2.0), (0.2, 0.8)]
labels = [
    '① A>0, r>1+ε
(好动作,过度更新 → 裁剪)',
    '③ A>0, r<1-ε
(好动作,误减 → 小奖励)',
    '② A<0, r>1+ε
(坏动作,错增 → 大惩罚)',
    '④ A<0, r<1-ε
(坏动作,过度减少 → 裁剪)',
]
colors = ['#2ecc71', '#27ae60', '#e74c3c', '#c0392b']
positions = [(0.3, 0.65), (0.3, 0.15), (0.7, 0.65), (0.7, 0.15)]

for (A_val, r_range, label, color, pos) in zip(A_vals, r_ranges, labels, colors, positions):
    r_example = np.linspace(r_range[0], r_range[1], 100)
    surr1 = r_example * A_val
    surr2 = np.clip(r_example, 1-eps, 1+eps) * A_val
    L_clip = np.minimum(surr1, surr2)
    ax.plot(r_example, L_clip, color=color, linewidth=3)
    ax.annotate(label, xy=(r_example[len(r_example)//2], L_clip[len(L_clip)//2]),
                fontsize=9, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

# 背景标记
ax.axvline(1-eps, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1+eps, color='gray', linestyle=':', alpha=0.5)
ax.axvline(1.0, color='black', linestyle='-', alpha=0.2)
ax.axhline(0, color='black', linestyle='-', linewidth=0.5, alpha=0.3)
ax.set_xlabel('比率 r_t(θ)'); ax.set_ylabel('L^CLIP')
ax.set_title('PPO-Clip 的四种行为模式')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('outputs/figures/19_ppo_four_cases.png', dpi=100)
plt.close()
print("✅ 图已保存到 outputs/figures/19_ppo_four_cases.png")

print("\n可视化完成！共生成 3 张图。")


## PPO 实现注意事项

### 常见陷阱

| 陷阱 | 问题 | 解决方案 |
|:----:|:-----|:---------|
| **Advantage 不归一化** | 优势值尺度不一致 | 在 minibatch 上标准化优势：$A \leftarrow (A - \bar{A}) / \sigma_A$ |
| **学习率过高** | clip fraction > 0.5 | 降低 lr，或使用学习率调度器 |
| **batch_size 太小** | 梯度噪声大 | 增大 batch_size (64-512) |
| **价值网络过拟合** | explained_var 下降 | 减少 n_epochs 或增大网络容量 |
| **熵损失权重不当** | 策略过早确定性 | 监控 entropy，$c_2 \in [0.01, 0.05]$ |
| **GAE 计算顺序错误** | 优势值错误 | 必须从后向前递推计算 GAE |

### 推荐的实现流程

```
1. 收集 n_steps 步数据 (state, action, reward, done, log_prob, value)
2. 计算 GAE 优势 (从后向前)
3. 标准化优势 (mean=0, std=1)
4. 多 epoch minibatch 更新:
   a. 计算新旧策略比率 r = exp(log_prob_new - log_prob_old)
   b. 计算裁剪损失 L^CLIP = min(r·A, clip(r)·A)
   c. 计算价值损失 (MSE with clipping)
   d. 总损失 = L^CLIP + c1·L^VF + c2·(-entropy)
   e. 梯度裁剪 (max_grad_norm=0.5)
5. 可选: KL 提前停止 (target_kl=0.01)
6. 清空缓冲区，回到步骤 1
```

### 调试工具

```python
# 核心检查
assert 0 < clip_fraction < 0.5, f"裁剪分数异常: {clip_fraction}"
assert approx_kl < 0.1, f"KL 散度过大: {approx_kl}"
assert explained_var > -0.5, f"解释方差异常: {explained_var}"

# 检查比率分布
ratios = torch.exp(new_log_probs - old_log_probs)
print(f"比率: mean={ratios.mean():.3f}, std={ratios.std():.3f}, "
      f"min={ratios.min():.3f}, max={ratios.max():.3f}")
```

### 性能优化

1. **向量化计算**：使用 PyTorch 批量计算，避免逐时间步的 for 循环
2. **GAE 矢量化**：使用卷积或累加和实现 O(T) GAE
3. **多环境并行**：同时运行多个环境收集数据，提高 GPU 利用率
4. **混合精度**：使用 fp16 减少显存和加速计算


## 本节总结

### 核心公式回顾

**Surrogate Objective (CPI)**:
$$L^{\text{CPI}}(\theta) = \mathbb{E}_t\left[\frac{\pi_\theta(a_t \mid s_t)}{\pi_{\theta_{\text{old}}}(a_t \mid s_t)} \, A_t\right]$$

**PPO-Clip**:
$$L^{\text{CLIP}}(\theta) = \mathbb{E}_t\left[\min\left(r_t(\theta) A_t, \; \text{clip}(r_t(\theta), 1-\varepsilon, 1+\varepsilon) \, A_t\right)\right]$$

**Total Loss**:
$$L_t(\theta) = \mathbb{E}_t\left[L^{\text{CLIP}}(\theta) - c_1 L^{\text{VF}}(\theta) + c_2 H(\pi_\theta(\cdot \mid s_t))\right]$$

### PPO 的关键贡献

| 创新 | 解决的问题 | 效果 |
|:----:|:-----------|:-----|
| **裁剪目标函数** | TRPO 复杂度高 | 一阶优化，实现简单 |
| **多 epoch 更新** | 样本效率低 | 数据利用率提高 5-10x |
| **组合损失** | 策略不稳定性 | 同时优化策略、价值和探索 |

### PPO vs TRPO

| 对比维度 | TRPO | PPO |
|:--------:|:----:|:---:|
| 约束方式 | KL 约束 | 裁剪 / KL 惩罚 |
| 梯度信息 | 二阶（Fisher） | 一阶 |
| 实现复杂度 | 高 | 低 |
| 更新速度 | 慢 | 快 |
| 样本效率 | 低 | 高（多 epoch）|
| 稳定性 | 非常稳定 | 稳定 |

### 核心理解

> PPO 的成功不在于它发明了全新的理论，而在于它用一个极其简单的技巧（裁剪）实现了与复杂方法（TRPO）同样甚至更好的效果。这种"简单即美"的设计哲学使得 PPO 成为了深度 RL 的标配算法。


## 练习

1. **裁剪公式推导**：证明 $\min(rA, \text{clip}(r, 1-\varepsilon, 1+\varepsilon) A)$ 等价于分段函数分析中的四种情况。

2. **$\varepsilon$ 敏感性**：修改可视化代码，分别设 $\varepsilon = 0.0, 0.1, 0.2, 0.5, 1.0$，观察裁剪函数形状的变化。讨论 $\varepsilon$ 取极端值（0 和 1）时 PPO 退化成什么算法。

3. **Clip Fraction 实验**：实现一个实验，在不同学习率下训练 PPO 并记录 clip fraction。绘制学习率 vs clip fraction 的关系图。

4. **实现 PPO-Penalty**：基于 PPO 的代码实现 PPO-Penalty 变体，比较 PPO-Clip 和 PPO-Penalty 在 CartPole 上的表现。

5. **裁剪 vs 信任区域**：设计一个简单场景（如 2 动作 MDP），对比 PPO 的裁剪和 TRPO 的 KL 约束在策略更新上的实际差异。

6. **多 epoch 的极限**：逐渐增加 `n_epochs`（如 1, 3, 10, 30, 100），观察性能变化。为什么 `n_epochs` 不是越大越好？
